# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Dictionary unpacking** (for new column names with spaces): `.pt.mutate(**{"body mass kg": "body_mass_g / 1000"})`
3. **External file specification**: `.pt.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
40,Adelie,3150.0,36.5,2.364421
195,Chinstrap,3500.0,45.5,1.690617
327,Gentoo,5500.0,53.4,1.928769
180,Chinstrap,3700.0,46.4,1.718564
54,Adelie,2900.0,34.5,2.436463
343,Gentoo,5400.0,49.9,2.168666
80,Adelie,3200.0,34.6,2.672993
232,Gentoo,4650.0,45.5,2.246106
224,Gentoo,5400.0,47.6,2.383306
259,Gentoo,5350.0,48.7,2.255775


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
308,Gentoo,4875.0,4.875,True
164,Chinstrap,3700.0,3.700,False
85,Adelie,3550.0,3.550,False
103,Adelie,4250.0,4.250,True
195,Chinstrap,3500.0,3.500,False
169,Chinstrap,3700.0,3.700,False
258,Gentoo,4350.0,4.350,True
40,Adelie,3150.0,3.150,False
48,Adelie,3450.0,3.450,False
106,Adelie,3750.0,3.750,False


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
296,Gentoo,4600.0,4.600,10.141252
12,Adelie,3200.0,3.200,7.054784
4,Adelie,3450.0,3.450,7.605939
16,Adelie,3450.0,3.450,7.605939
238,Gentoo,4800.0,4.800,10.582176
274,Gentoo,4900.0,4.900,10.802638
314,Gentoo,4850.0,4.850,10.692407
117,Adelie,3775.0,3.775,8.322440
221,Gentoo,5700.0,5.700,12.566334
99,Adelie,4100.0,4.100,9.038942


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
336,Gentoo,False
99,Adelie,True
7,Adelie,True
338,Gentoo,False
312,Gentoo,False
146,Adelie,True
189,Chinstrap,False
167,Chinstrap,False
26,Adelie,True
274,Gentoo,False


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
224,Gentoo,5400.0,True
166,Chinstrap,3575.0,False
14,Adelie,4400.0,True
215,Chinstrap,4000.0,True
189,Chinstrap,4800.0,True
309,Gentoo,5550.0,True
107,Adelie,3900.0,False
305,Gentoo,5600.0,True
282,Gentoo,4400.0,True
239,Gentoo,5350.0,True


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
118,Adelie,3.350
234,Gentoo,4.200
154,Chinstrap,3.650
257,Gentoo,5.250
42,Adelie,3.100
34,Adelie,3.325
215,Chinstrap,4.000
57,Adelie,3.800
23,Adelie,3.950
161,Chinstrap,3.700


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Backticks protect column names that contain spaces
spaced.pt.mutate(bmi="`body mass g` / `bill length mm` ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
16,Adelie,Torgersen,3450.0,38.7,2.303547
332,Gentoo,Biscoe,4650.0,43.5,2.457392
147,Adelie,Dream,3475.0,36.6,2.594135
66,Adelie,Biscoe,3350.0,35.5,2.658203
258,Gentoo,Biscoe,4350.0,44.0,2.246901
240,Gentoo,Biscoe,5700.0,50.2,2.261869
262,Gentoo,Biscoe,4300.0,45.3,2.095425
80,Adelie,Torgersen,3200.0,34.6,2.672993
60,Adelie,Biscoe,3150.0,35.7,2.471577
237,Gentoo,Biscoe,6300.0,49.2,2.602617


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
85,Adelie,3550.0,light
45,Adelie,4600.0,heavy
16,Adelie,3450.0,light
262,Gentoo,4300.0,heavy
144,Adelie,3000.0,light
113,Adelie,4275.0,heavy
39,Adelie,4650.0,heavy
36,Adelie,3950.0,light
205,Chinstrap,4050.0,heavy
194,Chinstrap,3550.0,light


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
161,Chinstrap,3700.0,medium
283,Gentoo,5650.0,large
276,Gentoo,4300.0,medium
303,Gentoo,5350.0,large
92,Adelie,3400.0,small
14,Adelie,4400.0,medium
273,Gentoo,5000.0,large
108,Adelie,3175.0,small
33,Adelie,3900.0,medium
52,Adelie,3450.0,small


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
21,Adelie,Biscoe,BIS
172,Chinstrap,Dream,OTH
252,Gentoo,Biscoe,BIS
261,Gentoo,Biscoe,BIS
125,Adelie,Torgersen,TOR
218,Chinstrap,Dream,OTH
97,Adelie,Dream,OTH
16,Adelie,Torgersen,TOR
300,Gentoo,Biscoe,BIS
188,Chinstrap,Dream,OTH


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [16]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
242,Gentoo,4.4,9.700328
61,Adelie,4.4,9.700328
43,Adelie,4.4,9.700328
62,Adelie,3.6,7.936632
10,Adelie,3.3,7.275246


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [17]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


## Creating new columns with spaces — dictionary unpacking (`**{...}`)

Python keyword argument syntax requires valid identifier names (`key=value`), so bare names with spaces like `my col = "..."` cause a Python `SyntaxError`.

To create a new column whose name contains spaces, pass the column definition in a dictionary and unpack it using `**`:


In [18]:
# Create a new column with spaces using dictionary unpacking
(
    spaced
    .pt.mutate(**{'body mass ratio': '[body mass g] / [bill length mm]'})
)


,body mass g,bill length mm,body mass ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


In [19]:
# Define multiple new columns with spaces at once
(
    penguins
    .pt.mutate(**{
        'body mass kg': 'body_mass_g / 1000',
        'bill ratio': 'bill_length_mm / bill_depth_mm',
    })
    .pt.select('species', 'body mass kg', 'bill ratio')
    .head()
)


,species,body mass kg,bill ratio
0,Adelie,3.75,2.090909
1,Adelie,3.80,2.270115
2,Adelie,3.25,2.238889
3,Adelie,NaN,NaN
4,Adelie,3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```
